# Day 9 — RAG Integration & Testing (ChromaDB + Fine-Tuned LLM Pipeline)

Welcome to the **Day 9 RAG Pipeline Integration & Testing Notebook** (Jira **KAN-49**)! In this session, we will:
1. **Set up ChromaDB Vector Database** using `all-MiniLM-L6-v2` embeddings.
2. **Index enterprise domain documents** (Retail Support Policies and Lean Six Sigma Manufacturing SOPs).
3. **Test the Full RAG Pipeline live**: User Query ➔ ChromaDB Semantic Search ➔ Context Injection ➔ Fine-Tuned Llama v3 / Qwen v3 Generation.
4. **Benchmark RAG performance on 200 test queries** calculating BLEU, ROUGE-1, ROUGE-2, and ROUGE-L.
5. Plot comparative performance metrics and sync ChromaDB vector database and evaluation reports directly to Google Drive.

---  
## Step 1: Mount Google Drive & Install Required Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q chromadb sentence-transformers transformers datasets accelerate peft bitsandbytes rouge-score nltk pandas matplotlib huggingface_hub

---  
## Step 2: Initialize Workspace & Write RAG Code and Knowledge Base

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Project folder not found on Drive. Defaulting target back to: {gdrive_dir}")
else:
    print(f"[+] Detected active Drive folder: {gdrive_dir}")

for d in ["data/raw", "data/processed", "data/knowledge_base", "data/chroma_db", "configs", "src", "models/evaluation"]:
    os.makedirs(os.path.join(project_dir, d), exist_ok=True)

drive_processed = os.path.join(gdrive_dir, "data", "processed")
local_processed = os.path.join(project_dir, "data", "processed")
if os.path.isdir(drive_processed):
    print("[*] Copying data splits from Google Drive...")
    !cp -v "{drive_processed}/"*.json "{local_processed}/" 2>/dev/null || true

# Write Knowledge Base Documents
retail_kb = "# Enterprise Retail & E-Commerce Customer Support Policies Manual\n\n## Section 1: Order Cancellation & Modification Policy\n- **Cancellation Window**: Customers can cancel orders within 60 minutes of placement directly from their account dashboard or by contacting customer support with their {{Order Number}}.\n- **Post-Dispatch Policy**: If an order has already entered the fulfillment or dispatched state, it cannot be canceled. The customer must wait for delivery and initiate a standard return.\n- **Address Changes**: Shipping address updates are permitted only while the order status is \"Processing\" and before label creation. Once tracking is generated, address modifications must be requested directly through the shipping courier (FedEx, UPS, DHL).\n\n## Section 2: Returns, Exchanges & Refunds\n- **Return Period**: We offer a 30-day return policy from the date of package delivery for all unworn, unused items with original tags and packaging intact.\n- **Return Initiation**: Customers must provide their {{Order Number}}, verify their {{Client Last Name}}, and select the return reason via the online portal to receive a prepaid return shipping label.\n- **Refund Processing Time**: Once the returned item is inspected at our fulfillment center, refunds are processed within 3 to 5 business days back to the original payment method (Credit Card, PayPal, or Bank Transfer). Store credit refunds are issued immediately upon return scanning.\n- **Damaged or Defective Items**: If an item arrives damaged or incorrect, customers must report the issue within 48 hours of delivery along with photographic evidence. Immediate free replacements or full refunds are provided without restocking fees.\n\n## Section 3: Accepted Payment Methods & Billing Inquiries\n- **Accepted Payment Gateways**: We accept Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and direct Wire/Bank Transfers.\n- **Payment Errors & Failed Transactions**: In the event of double charges or failed payment gateway authorizations, the pending authorization holds typically drop from the customer's bank statement within 24 to 72 hours.\n- **Invoice & Receipt Requests**: Detailed tax invoices with VAT/GST breakdowns are automatically sent to the registered email address and can also be downloaded from the \"Order History\" section.\n\n## Section 4: Shipping, Delivery & Tracking Guidelines\n- **Standard Shipping**: 3 to 5 business days delivery across all domestic zones.\n- **Express / Expedited Shipping**: 1 to 2 business days delivery with priority courier handling.\n- **Tracking Inquiries**: Customers can track package locations in real-time using their {{Tracking Number}} on our tracking portal or the carrier's official website.\n- **Delayed Shipments**: If tracking shows no movement for more than 4 business days, customer support will file a trace with the carrier and offer expedited reshipment or compensation credits.\n\n## Section 5: Account Security & Profile Management\n- **Password Reset & Account Recovery**: Customers can request a secure password reset link sent to their verified email. Multi-Factor Authentication (MFA) is recommended for all account tiers.\n- **Account Deletion / GDPR Requests**: Requests to permanently remove account data can be submitted through the Privacy Settings tab and are executed within 14 business days in compliance with data privacy regulations.\n";
with open("/content/Retail/data/knowledge_base/retail_ecommerce_policies.md", "w", encoding="utf-8") as f:
    f.write(retail_kb)
print("[+] data/knowledge_base/retail_ecommerce_policies.md written.")

mfg_kb = "# Manufacturing Operations & Quality Control Standard Operating Procedures (SOP)\n\n## Section 1: Lean Six Sigma DMAIC Methodology\n- **Define Phase**: Explicitly identify the project scope, problem statement, customer CTQ (Critical to Quality) requirements, and form the cross-functional Project Charter.\n- **Measure Phase**: Establish baseline process capabilities, validate measurement systems using Gage R&R (Repeatability and Reproducibility < 10%), and collect accurate defect frequency data.\n- **Analyze Phase**: Identify root causes of variation or defect spikes using Cause-and-Effect (Fishbone/Ishikawa) diagrams, Pareto 80/20 analysis, and the 5 Whys interrogation technique.\n- **Improve Phase**: Design and implement targeted countermeasures, conduct Design of Experiments (DOE), streamline workflows with Kaizen events, and eliminate process bottlenecks.\n- **Control Phase**: Standardize the optimized process through updated SOPs, implement Poka-Yoke (mistake-proofing) mechanisms, and establish Statistical Process Control (SPC) monitoring plans.\n\n## Section 2: Statistical Process Control (SPC) & Control Charts\n- **X-bar and R Charts**: Used for continuous variable data collected in subgroups (e.g., component thickness, diameter, tensile strength). The X-bar chart tracks process average/central tendency, while the R chart tracks process dispersion/variation.\n- **P-Charts and C-Charts**: Used for attribute/discrete defect data. P-charts monitor the proportion of nonconforming units, while C-charts monitor the total count of defects per unit.\n- **Control Limits (UCL / LCL)**: Statistically calculated at +/- 3 standard deviations (+/- 3 Sigma) from the central line. Points falling beyond control limits or exhibiting non-random trends (e.g., 7 consecutive points on one side of the center line) indicate assignable causes that require immediate line pauses.\n\n## Section 3: Assembly Line Calibration & Maintenance Procedures\n- **Daily Calibration Protocol**: All optical measurement sensors, torque wrenches, and thermal bonding units must undergo zero-point calibration before shift commencement.\n- **Out-of-Specification Escalation**: If equipment drifts beyond +/- 0.05mm tolerance, the line operator must immediately tag the machine \"Out of Service\", notify the Shift Quality Lead, and quarantine all parts manufactured in the preceding 60-minute window.\n- **Preventative Maintenance (TPM)**: Total Productive Maintenance schedules dictate autonomous daily lubrication, weekly pneumatic pressure checks, and monthly sensor recalibration.\n\n## Section 4: Defect Root Cause Analysis (RCA) & Corrective Actions\n- **5 Whys Protocol**: For every critical non-conformance, trace the failure mode backward through at least 5 iterative \"Why\" queries to uncover the organizational or systemic failure rather than operator error.\n- **8D Problem Solving**: Used for major customer escapes or recurring defects. Follows the 8 disciplines from emergency response (D3) to permanent corrective actions (D5) and systemic prevention (D7).\n- **Corrective and Preventive Action (CAPA)**: Document all containment actions within 24 hours and establish root-cause validation within 5 business days.\n";
with open("/content/Retail/data/knowledge_base/manufacturing_sop_manual.md", "w", encoding="utf-8") as f:
    f.write(mfg_kb)
print("[+] data/knowledge_base/manufacturing_sop_manual.md written.")

# Write Pipeline & Evaluation scripts
rag_pipe_code = "import os\nimport re\nimport glob\nimport torch\nimport chromadb\nfrom chromadb.utils import embedding_functions\n\nclass RetailMfgRAGPipeline:\n    \"\"\"\n    Production-grade RAG pipeline using ChromaDB for vector retrieval\n    and fine-tuned QLoRA models (Llama-3 / Qwen) for domain-grounded generation.\n    \"\"\"\n    \n    def __init__(self, persist_dir=\"data/chroma_db\", collection_name=\"retail_mfg_knowledge\", embedding_model=\"all-MiniLM-L6-v2\"):\n        self.persist_dir = persist_dir\n        self.collection_name = collection_name\n        os.makedirs(self.persist_dir, exist_ok=True)\n        \n        print(f\"[*] Initializing ChromaDB Client at: {self.persist_dir}\")\n        self.client = chromadb.PersistentClient(path=self.persist_dir)\n        \n        # Initialize Sentence Transformer Embedding Function\n        print(f\"[*] Loading Embedding Model: {embedding_model}...\")\n        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(\n            model_name=embedding_model\n        )\n        \n        self.collection = self.client.get_or_create_collection(\n            name=self.collection_name,\n            embedding_function=self.embedding_fn,\n            metadata={\"hnsw:space\": \"cosine\"}\n        )\n        print(f\"[+] Collection '{self.collection_name}' ready. Current indexed documents: {self.collection.count()}\")\n\n    def chunk_markdown_document(self, file_path):\n        \"\"\"\n        Parses and chunks markdown documents by sections and bullet points.\n        \"\"\"\n        chunks = []\n        with open(file_path, \"r\", encoding=\"utf-8\") as f:\n            content = f.read()\n            \n        file_basename = os.path.basename(file_path)\n        domain = \"manufacturing\" if \"manufacturing\" in file_basename.lower() or \"sop\" in file_basename.lower() else \"retail\"\n        \n        # Split by sections (## Section)\n        sections = re.split(r'\\n(?=##\\s+)', content)\n        for sec in sections:\n            sec = sec.strip()\n            if not sec:\n                continue\n                \n            lines = sec.split(\"\\n\")\n            section_title = lines[0].replace(\"#\", \"\").strip()\n            \n            # Split section into individual bullet items or paragraphs\n            body_paragraphs = [p.strip() for p in sec.split(\"\\n- \") if p.strip()]\n            \n            for idx, para in enumerate(body_paragraphs):\n                # Clean header from the first bullet if attached\n                if idx == 0 and \"\\n\" in para:\n                    para_parts = para.split(\"\\n\", 1)\n                    if len(para_parts) > 1 and para_parts[1].strip():\n                        para = para_parts[1].strip()\n                        \n                clean_text = para.replace(\"- \", \"\").replace(\"**\", \"\").strip()\n                if len(clean_text) > 30:\n                    prefix = \"Process Operations Manual: \" if domain == \"manufacturing\" else \"Customer Support Policy Guide: \"\n                    formatted_chunk = f\"{prefix}{section_title} - {clean_text}\"\n                    \n                    chunk_id = f\"{file_basename}_{section_title[:20]}_{idx}\".replace(\" \", \"_\").replace(\"/\", \"_\")\n                    chunks.append({\n                        \"id\": chunk_id,\n                        \"text\": formatted_chunk,\n                        \"metadata\": {\n                            \"source\": file_basename,\n                            \"section\": section_title,\n                            \"domain\": domain\n                        }\n                    })\n        return chunks\n\n    def index_knowledge_directory(self, kb_dir=\"data/knowledge_base\"):\n        \"\"\"\n        Indexes all markdown and text documents from knowledge base directory into ChromaDB.\n        \"\"\"\n        doc_files = glob.glob(os.path.join(kb_dir, \"*.md\")) + glob.glob(os.path.join(kb_dir, \"*.txt\"))\n        if not doc_files:\n            print(f\"[!] Warning: No knowledge base files found in {kb_dir}\")\n            return 0\n            \n        all_chunks = []\n        for fpath in doc_files:\n            print(f\"[*] Parsing knowledge file: {fpath}...\")\n            chunks = self.chunk_markdown_document(fpath)\n            all_chunks.extend(chunks)\n            \n        if all_chunks:\n            # Upsert into ChromaDB\n            ids = [c[\"id\"] for c in all_chunks]\n            documents = [c[\"text\"] for c in all_chunks]\n            metadatas = [c[\"metadata\"] for c in all_chunks]\n            \n            self.collection.upsert(\n                ids=ids,\n                documents=documents,\n                metadatas=metadatas\n            )\n            print(f\"[+] Successfully indexed {len(all_chunks)} knowledge passages into ChromaDB!\")\n            print(f\"    - Total database records: {self.collection.count()}\")\n            \n        return len(all_chunks)\n\n    def retrieve_context(self, query, top_k=2):\n        \"\"\"\n        Retrieves top_k most relevant knowledge passages for a given user query.\n        \"\"\"\n        results = self.collection.query(\n            query_texts=[query],\n            n_results=top_k\n        )\n        \n        retrieved_docs = results[\"documents\"][0] if results[\"documents\"] else []\n        retrieved_metas = results[\"metadatas\"][0] if results[\"metadatas\"] else []\n        \n        combined_context = \" \".join(retrieved_docs)\n        return combined_context, retrieved_docs, retrieved_metas\n\n    def generate_rag_response(self, model, tokenizer, query, top_k=2, max_new_tokens=150, temperature=0.7):\n        \"\"\"\n        Retrieves relevant context and generates a grounded response using the fine-tuned LLM.\n        \"\"\"\n        context_str, raw_docs, metadatas = self.retrieve_context(query, top_k=top_k)\n        \n        # Build Alpaca RAG Prompt\n        prompt = (\n            f\"Below is an instruction that describes a task, paired with an input that provides further context. \"\n            f\"Write a response that appropriately completes the request.\\n\\n\"\n            f\"### Instruction:\\n{query}\\n\\n\"\n            f\"### Context:\\n{context_str}\\n\\n\"\n            f\"### Response:\\n\"\n        )\n        \n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n        prompt_len = inputs.input_ids.shape[1]\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=max_new_tokens,\n                temperature=temperature,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n            \n        generation_tokens = outputs[0][prompt_len:]\n        response = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        \n        return {\n            \"query\": query,\n            \"context\": context_str,\n            \"retrieved_passages\": raw_docs,\n            \"sources\": [m.get(\"source\", \"\") for m in metadatas],\n            \"response\": response\n        }\n\nif __name__ == \"__main__\":\n    # Test indexing standalone\n    pipeline = RetailMfgRAGPipeline()\n    pipeline.index_knowledge_directory(\"data/knowledge_base\")\n    \n    sample_query = \"What is the return window and refund time for damaged items?\"\n    ctx, docs, _ = pipeline.retrieve_context(sample_query, top_k=2)\n    print(f\"\\n[*] Sample Query: {sample_query}\")\n    print(f\"[*] Retrieved Context: {ctx}\")\n";
with open("/content/Retail/src/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write(rag_pipe_code)
print("[+] src/rag_pipeline.py written.")

eval_rag_code = "import os\nimport json\nimport random\nimport argparse\nimport torch\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\nfrom rouge_score import rouge_scorer\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig\nfrom peft import PeftModel\n\n# Import our RAG Pipeline\nfrom rag_pipeline import RetailMfgRAGPipeline\n\ntry:\n    nltk.download('punkt', quiet=True)\n    nltk.download('punkt_tab', quiet=True)\nexcept Exception:\n    pass\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate End-to-End RAG Pipeline on 200 Queries\")\n    parser.add_argument(\"--model_id\", type=str, default=\"meta-llama/Meta-Llama-3-8B-Instruct\", help=\"Base Foundation Model\")\n    parser.add_argument(\"--adapter_dir\", type=str, required=True, help=\"Path to Fine-Tuned v3 LoRA Adapter\")\n    parser.add_argument(\"--test_file\", type=str, default=\"data/processed/test_v3.json\", help=\"Test dataset path\")\n    parser.add_argument(\"--kb_dir\", type=str, default=\"data/knowledge_base\", help=\"Knowledge base documents directory\")\n    parser.add_argument(\"--chroma_dir\", type=str, default=\"data/chroma_db\", help=\"ChromaDB persistence directory\")\n    parser.add_argument(\"--output_file\", type=str, default=\"models/evaluation/rag_pipeline_200_results.json\", help=\"Output results path\")\n    parser.add_argument(\"--num_samples\", type=int, default=200, help=\"Number of test queries to benchmark\")\n    parser.add_argument(\"--seed\", type=int, default=42, help=\"Random seed for sampling\")\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    \n    print(\"\\n=======================================================\")\n    print(\"[*] DAY 9: End-to-End RAG Pipeline Benchmark (200 Queries)\")\n    print(f\"[*] Base Model:      {args.model_id}\")\n    print(f\"[*] Adapter Path:    {args.adapter_dir}\")\n    print(f\"[*] Test Dataset:    {args.test_file}\")\n    print(f\"[*] Number of Tests: {args.num_samples}\")\n    print(\"=======================================================\\n\")\n    \n    # 1. Initialize & Index Vector Database\n    rag_pipe = RetailMfgRAGPipeline(persist_dir=args.chroma_dir)\n    rag_pipe.index_knowledge_directory(args.kb_dir)\n    \n    token = os.environ.get(\"HF_TOKEN\")\n    if not token:\n        try:\n            from huggingface_hub import get_token\n            token = get_token()\n        except Exception:\n            token = None\n    \n    # 2. Load Model & Tokenizer\n    print(\"[*] Loading Tokenizer...\")\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True, token=token)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    print(\"[*] Loading Model in 4-bit Quantization...\")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16\n    )\n    \n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16,\n        token=token\n    )\n    \n    for name, param in base_model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in base_model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    print(f\"[*] Attaching Fine-Tuned v3 LoRA Adapter from: {args.adapter_dir}...\")\n    model = PeftModel.from_pretrained(base_model, args.adapter_dir)\n    model.eval()\n    \n    # 3. Load Test Data\n    if not os.path.exists(args.test_file):\n        # Fallback to test.json if test_v3.json not directly in local path\n        fallback = args.test_file.replace(\"_v3\", \"\")\n        if os.path.exists(fallback):\n            args.test_file = fallback\n            \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n        \n    if len(test_data) > args.num_samples:\n        print(f\"[*] Randomly sampling {args.num_samples} evaluation queries from {len(test_data)} available.\")\n        eval_samples = random.sample(test_data, args.num_samples)\n    else:\n        print(f\"[*] Evaluating all {len(test_data)} test queries.\")\n        eval_samples = test_data\n\n    # 4. Scorers\n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    print(\"\\n[*] Running RAG Retrieval + LLM Generation...\")\n    for idx, sample in enumerate(eval_samples):\n        query = sample.get(\"instruction\", \"\")\n        reference = sample.get(\"response\", \"\")\n        \n        # End-to-end RAG call\n        rag_output = rag_pipe.generate_rag_response(\n            model=model,\n            tokenizer=tokenizer,\n            query=query,\n            top_k=2,\n            max_new_tokens=150\n        )\n        prediction = rag_output[\"response\"]\n        retrieved_context = rag_output[\"context\"]\n        \n        # Metric Calculations\n        rouge_scores = rouge_scorer_inst.score(reference, prediction)\n        r1 = rouge_scores['rouge1'].fmeasure\n        r2 = rouge_scores['rouge2'].fmeasure\n        rl = rouge_scores['rougeL'].fmeasure\n        \n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(prediction.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"query\": query,\n            \"retrieved_context\": retrieved_context,\n            \"reference\": reference,\n            \"prediction\": prediction,\n            \"sources\": rag_output[\"sources\"],\n            \"rouge1\": r1,\n            \"rouge2\": r2,\n            \"rougeL\": rl,\n            \"bleu\": bleu\n        })\n        \n        if (idx + 1) % 25 == 0 or (idx + 1) == len(eval_samples):\n            print(f\"    - Processed {idx + 1}/{len(eval_samples)} queries | Current Avg BLEU: {total_bleu/(idx+1):.4f}\")\n\n    n = len(eval_samples)\n    summary = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": args.adapter_dir,\n        \"total_evaluated\": n,\n        \"mean_rouge1\": total_r1 / n,\n        \"mean_rouge2\": total_r2 / n,\n        \"mean_rougeL\": total_rl / n,\n        \"mean_bleu\": total_bleu / n\n    }\n    \n    print(\"\\n=================== RAG EVALUATION SUMMARY (200 QUERIES) ===================\")\n    print(f\"[+] Mean ROUGE-1: {summary['mean_rouge1']:.4f}\")\n    print(f\"[+] Mean ROUGE-2: {summary['mean_rouge2']:.4f}\")\n    print(f\"[+] Mean ROUGE-L: {summary['mean_rougeL']:.4f}\")\n    print(f\"[+] Mean BLEU:    {summary['mean_bleu']:.4f}\")\n    print(\"============================================================================\\n\")\n    \n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump({\"summary\": summary, \"results\": results}, f, indent=2, ensure_ascii=False)\n        \n    print(f\"[+] Detailed RAG evaluation report saved to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate_rag.py", "w", encoding="utf-8") as f:
    f.write(eval_rag_code)
print("[+] src/evaluate_rag.py written.")

---  
## Step 3: Index Knowledge Base into ChromaDB Vector Database

In [ ]:
import sys
sys.path.append('/content/Retail/src')
from rag_pipeline import RetailMfgRAGPipeline

pipeline = RetailMfgRAGPipeline(persist_dir="/content/Retail/data/chroma_db")
num_chunks = pipeline.index_knowledge_directory("/content/Retail/data/knowledge_base")
print(f"\n[+] ChromaDB vector database successfully initialized with {num_chunks} enterprise knowledge chunks!")

---  
## Step 4: Live Interactive RAG Query Demo (Semantic Retrieval Test)

In [ ]:
sample_queries = [
    "How many days do I have to return an order, and how long does the refund take?",
    "What are the 5 phases of Lean Six Sigma DMAIC and how do we monitor control charts?",
    "Can I cancel my order after it has been dispatched from the warehouse?",
    "What is the protocol if an assembly line sensor drifts out of specification tolerance?"
]

print("=================== CHROMADB SEMANTIC RETRIEVAL TEST ===================\n")
for q in sample_queries:
    ctx, docs, metas = pipeline.retrieve_context(q, top_k=2)
    print(f"🔹 User Query: {q}")
    print(f"   📚 Source Manual: {metas[0]['source']} (Section: {metas[0]['section']})")
    print(f"   🔍 Retrieved Passage: {docs[0][:140]}...")
    print("-" * 70 + "\n")

---  
## Step 5: Execute 200-Query Automated RAG Benchmark (Llama v3 RAG Pipeline)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

hf_token_val = None
for secret_name in ['HF_TOKEN', 'HF_TOKEI', 'HF_token', 'hf_token']:
    try:
        val = userdata.get(secret_name)
        if val:
            hf_token_val = val
            login(token=val, add_to_git_credential=False)
            os.environ['HF_TOKEN'] = val
            print(f"[+] Successfully logged into Hugging Face via {secret_name}!")
            break
    except Exception:
        pass

!python /content/Retail/src/evaluate_rag.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v3" \
    --test_file /content/Retail/data/processed/test_v3.json \
    --kb_dir /content/Retail/data/knowledge_base \
    --chroma_dir /content/Retail/data/chroma_db \
    --output_file /content/Retail/models/evaluation/rag_pipeline_200_results.json \
    --num_samples 200

---  
## Step 6: Plot RAG Performance Comparison & View Grounded Predictions

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import os

eval_dir = "/content/Retail/models/evaluation"
rag_results_path = os.path.join(eval_dir, "rag_pipeline_200_results.json")

if os.path.exists(rag_results_path):
    with open(rag_results_path, "r") as f:
        rag_data = json.load(f)

    summary = rag_data["summary"]
    print("\n===================== FULL RAG PIPELINE (200 QUERIES) =====================")
    print(f"Total Queries Evaluated: {summary['total_evaluated']}")
    print(f"Mean ROUGE-1:            {summary['mean_rouge1']:.4f}")
    print(f"Mean ROUGE-2:            {summary['mean_rouge2']:.4f}")
    print(f"Mean ROUGE-L:            {summary['mean_rougeL']:.4f}")
    print(f"Mean BLEU:               {summary['mean_bleu']:.4f}")
    print("============================================================================\n")

    # Show Sample Grounded Predictions
    sample_records = rag_data["results"][:5]
    sample_df = pd.DataFrame([
        {
            "Query": r["query"],
            "Retrieved Passage (ChromaDB)": r["retrieved_context"],
            "RAG Model Answer": r["prediction"],
            "Ground Truth Reference": r["reference"],
            "BLEU": round(r["bleu"], 4)
        }
        for r in sample_records
    ])
    pd.set_option('display.max_colwidth', None)
    display(sample_df.style.set_properties(**{'text-align': 'left'}))

    # Comparison Bar Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
    scores = [summary["mean_rouge1"], summary["mean_rouge2"], summary["mean_rougeL"], summary["mean_bleu"]]
    bars = ax.bar(metrics, scores, color=['#2563eb', '#16a34a', '#d97706', '#9333ea'], width=0.5)
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 0.015, f"{yval:.3f}", ha='center', fontweight='bold')
    ax.set_ylim(0, 0.8)
    ax.set_title("Full RAG Pipeline Performance Benchmark (200 Test Queries)")
    ax.set_ylabel("Metric Score")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("[!] Run Step 5 first to generate RAG evaluation results.")

---  
## Step 7: Persist ChromaDB Index and RAG Evaluation Reports to Google Drive

In [ ]:
print(f"[*] Persisting ChromaDB database and RAG reports to Google Drive: {gdrive_dir}...")
drive_eval_dir = os.path.join(gdrive_dir, "models", "evaluation")
drive_chroma_dir = os.path.join(gdrive_dir, "data", "chroma_db")
os.makedirs(drive_eval_dir, exist_ok=True)
os.makedirs(drive_chroma_dir, exist_ok=True)

# Copy evaluation report
!cp -v /content/Retail/models/evaluation/rag_pipeline_200_results.json "{drive_eval_dir}/"

# Sync ChromaDB Vector Database
!rsync -av --progress /content/Retail/data/chroma_db/ "{drive_chroma_dir}/"
print("[+] Backup complete. ChromaDB index and evaluation benchmarks saved to Google Drive!")